# Laboratorium 4 - rekomendacje oparte na treści

## Przygotowanie

 * pobierz z Teamsów do katalogu `amazon/` (~27 MB): `Video_Games.{train,valid,test}.csv.gz`, `meta_Video_Games.jsonl.gz`
   * to przycięta wersja [Amazon Reviews 2023](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023) (Video Games, [5-core, podział czasowy](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/tree/main/benchmark/5core/timestamp) + [metadane](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/tree/main/raw/meta_categories)), skrypt: `prepare_data.py`
 * `pip install numpy pandas scipy scikit-learn nltk matplotlib sentence-transformers`
 * część 4. przy pierwszym uruchomieniu pobiera model (~90 MB) i liczy embeddingi (~10 min na CPU), wynik zapisuje na dysk
 * na słabszym sprzęcie ustaw `sample_items` i/lub `sample_users` w `Params` - `sample_items` skraca liczenie osadzeń, ale przerzedza historie (mniej użytkowników do oceny)

## Część 1. - dane

In [ ]:
import gzip
import json
import os
import re
from collections import Counter
from dataclasses import dataclass
from functools import lru_cache
from typing import Callable, Dict, List, Optional, Sequence, Set, Tuple

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd

from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
from nltk.stem import WordNetLemmatizer
from nltk.stem.lancaster import LancasterStemmer
from nltk.stem.porter import PorterStemmer
from nltk.stem.snowball import SnowballStemmer

from scipy import sparse
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

nltk.download('stopwords', quiet=True)

In [ ]:
@dataclass(frozen=True)
class Params:
    data_path: str = 'amazon'
    category: str = 'Video_Games'
    like_threshold: float = 4  # oceny >= progu traktujemy jako pozytywne
    min_history: int = 3  # min. liczba lubianych elementow w historii uzytkownika
    popularity_days: int = 365  # popularnosc liczymy z ostatnich dni historii
    embed_max_words: int = 100  # teksty dla modelu jezykowego skracamy do tylu slow
    eval_users: int = 2000  # losowa probka uzytkownikow do oceny
    ks: Tuple[int, ...] = (10, 50)  # dlugosci list rekomendacji (k w hit@k)
    sample_users: Optional[int] = None  # np. 20000 - pracujemy tylko na probce uzytkownikow
    sample_items: Optional[int] = None  # np. 5000 - pracujemy tylko na probce elementow
    seed: int = 42


PARAMS = Params()

Podział czasowy: `train` + `valid` to historia, `test` - późniejsze interakcje (jak w rzeczywistości - przyszłości nie znamy).
5-core: każdy użytkownik i element ma co najmniej 5 ocen.

In [ ]:
def load_interactions(params: Params) -> Tuple[pd.DataFrame, pd.DataFrame]:
    path = f'{params.data_path}/{params.category}'
    train = pd.concat([pd.read_csv(f'{path}.train.csv.gz'), pd.read_csv(f'{path}.valid.csv.gz')], ignore_index=True)
    test = pd.read_csv(f'{path}.test.csv.gz')
    return train, test


def sample_data(train: pd.DataFrame, test: pd.DataFrame, params: Params) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(params.seed)
    if params.sample_items:
        all_items = np.union1d(train.parent_asin.unique(), test.parent_asin.unique())
        kept = rng.choice(all_items, min(params.sample_items, len(all_items)), replace=False)
        train, test = train[train.parent_asin.isin(kept)], test[test.parent_asin.isin(kept)]
    if params.sample_users:
        all_users = train.user_id.unique()
        kept = rng.choice(all_users, min(params.sample_users, len(all_users)), replace=False)
        train, test = train[train.user_id.isin(kept)], test[test.user_id.isin(kept)]
    return train.copy(), test.copy()


def clean_text(text: str) -> str:
    text = re.sub(r'<[^>]+>', ' ', text)  # usuwamy znaczniki HTML
    return re.sub(r'\s+', ' ', text).strip()


def load_items(params: Params, item_ids: Set[str]) -> pd.DataFrame:
    rows = []
    with gzip.open(f'{params.data_path}/meta_{params.category}.jsonl.gz', 'rt', encoding='utf-8') as datafile:
        for line in datafile:
            meta = json.loads(line)
            if meta['parent_asin'] in item_ids:
                title = meta['title'] or ''
                text = ' '.join([title, *meta['features'], *meta['description']])
                rows.append({'item_id': meta['parent_asin'], 'title': title, 'text': clean_text(text)})
    return pd.DataFrame(rows)


train, test = sample_data(*load_interactions(PARAMS), PARAMS)
items = load_items(PARAMS, set(train.parent_asin) | set(test.parent_asin))
item_index = {item_id: i for i, item_id in enumerate(items.item_id)}
train['item'] = train.parent_asin.map(item_index)
test['item'] = test.parent_asin.map(item_index)
print(len(items), len(train), len(test))

In [ ]:
# zimny start - elementy z testu, ktore nie maja zadnej interakcji w historii
cold_items = set(test.item) - set(train.item)
print(f'nowe elementy w tescie: {len(cold_items) / test.item.nunique():.1%}')

## Część 2. - TF-IDF

In [ ]:
STOPWORDS = set(stopwords.words('english'))

def tokenize(text: str) -> List[str]:
    # usun wielkie litery, znaki specjalne, cyfry itp.
    # usun STOPWORDS
    # uzyj wybranego stemmera do przeksztalcenia slow w formy podstawowe
    raise NotImplementedError


corpus = [tokenize(text) for text in items.text]
print(items.text[0][:300])
print(corpus[0][:30])

$$\text{tfidf}(t, d) = \text{tf}(t, d) \cdot \left(\ln \frac{N}{\text{df}(t)} + 1\right)$$

 * $\text{tf}(t, d)$ - liczba wystąpień słowa $t$ w dokumencie $d$
 * $\text{df}(t)$ - liczba dokumentów zawierających $t$, $N$ - liczba dokumentów
 * $+1$ - by słowa obecne we wszystkich dokumentach nie dostały wagi 0
 * wektory dokumentów normalizujemy: $v_d \leftarrow \frac{v_d}{\lVert v_d \rVert_2}$

In [ ]:
# slownik: slowo -> indeks kolumny

def build_vocabulary(corpus: List[List[str]]) -> Dict[str, int]:
    # slowa posortowane alfabetycznie
    raise NotImplementedError


vocabulary = build_vocabulary(corpus)
print(len(vocabulary))

In [ ]:
# df(t) - w ilu dokumentach wystapilo slowo (kazdy dokument liczymy raz)

def document_frequency(corpus: List[List[str]], vocabulary: Dict[str, int]) -> np.ndarray:
    raise NotImplementedError


doc_freq = document_frequency(corpus, vocabulary)

In [ ]:
# macierz TF-IDF (dokumenty x slowa)

def tf_idf_matrix(corpus: List[List[str]], vocabulary: Dict[str, int], doc_freq: np.ndarray) -> sparse.csr_matrix:
    # zbierz trojki (wiersz, kolumna, wartosc) i zbuduj sparse.csr_matrix((values, (rows, cols)), shape=...)
    # normalize() normalizuje wiersze (L2), Counter zliczy slowa w dokumencie
    raise NotImplementedError


tfidf = tf_idf_matrix(corpus, vocabulary, doc_freq)
tfidf.shape

In [ ]:
# porownujemy z sklearn - roznica powinna wynosic ~0
# korpus jest juz stokenizowany, stad analyzer=lambda x: x

reference = TfidfVectorizer(analyzer=lambda x: x, vocabulary=vocabulary, smooth_idf=False).fit_transform(corpus)
print(abs(tfidf - reference).max())

## Część 3. - podobieństwo element-element

In [ ]:
# wektory sa znormalizowane, wiec iloczyn skalarny = podobienstwo kosinusowe

def scores(vectors, query) -> np.ndarray:
    result = vectors @ query.T
    result = result.toarray() if sparse.issparse(result) else np.asarray(result)
    return result.ravel()


def top_k(item_scores: np.ndarray, k: int, exclude: Sequence[int] = ()) -> List[int]:
    item_scores = item_scores.copy()
    item_scores[list(exclude)] = -np.inf
    k = min(k, len(item_scores) - 1)
    best = np.argpartition(-item_scores, k)[:k]
    return best[np.argsort(-item_scores[best])].tolist()


# k najpodobniejszych elementow, bez samego elementu

def most_similar(vectors, item: int, k: int) -> List[int]:
    # skorzystaj z scores i top_k
    raise NotImplementedError

In [ ]:
# funkcja znajdujaca teksty podobne do zadanej frazy

def find_items(phrase: str) -> pd.DataFrame:
    return items[items.title.str.contains(phrase, case=False, regex=False)].head()


def label(item: int, relevant: Set[int] = frozenset()) -> str:
    marks = ('+' if item in relevant else ' ') + ('*' if item in cold_items else ' ')
    return f'{marks} {items.title[item][:80]}'


def print_items(indices: Sequence[int], relevant: Set[int] = frozenset()):
    for i in indices:
        print(label(i, relevant))


find_items('Mario Kart 8')

In [ ]:
EXAMPLE_ITEMS = []
for phrase in ['Mario Kart 8', 'FIFA', 'Gaming Headset', 'Resident Evil']:
    found = find_items(phrase)
    if len(found):  # na probce fraza moze nic nie znalezc
        EXAMPLE_ITEMS.append(found.index[0])

# zauwaz: ile sposrod podobnych to inne edycje tej samej gry/sprzetu?
for item in EXAMPLE_ITEMS:
    print(items.title[item])
    print_items(most_similar(tfidf, item, 5))

## Część 4. - osadzenia z modelu językowego

In [ ]:
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
EMBEDDINGS_PATH = (
    f'embeddings_{MODEL_NAME.split("/")[-1]}_{PARAMS.embed_max_words}'
    f'_{PARAMS.sample_users}_{PARAMS.sample_items}_{PARAMS.seed}.npy'
)


# pierwsze uruchomienie: pobranie modelu i ~10 min liczenia na CPU, dlatego zapisujemy wynik na dysk

def embed(texts: List[str]) -> np.ndarray:
    if os.path.exists(EMBEDDINGS_PATH):
        embeddings = np.load(EMBEDDINGS_PATH)
        assert len(embeddings) == len(texts), f'nieaktualny plik {EMBEDDINGS_PATH} - usun go'
        return embeddings
    model = SentenceTransformer(MODEL_NAME)
    embeddings = model.encode(texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
    np.save(EMBEDDINGS_PATH, embeddings)
    return embeddings


# model i tak widzi tylko poczatek tekstu (max 256 tokenow), krotsze teksty licza sie szybciej
short_texts = [' '.join(text.split()[:PARAMS.embed_max_words]) for text in items.text]
llm = embed(short_texts)  # osadzenia z modelu jezykowego - jeden gesty wektor na element
llm.shape

Indeks Jaccarda - podobieństwo zbiorów:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|} \in [0, 1]$$

0 - zbiory rozłączne, 1 - identyczne.

In [ ]:
def jaccard(a: Sequence[int], b: Sequence[int]) -> float:
    raise NotImplementedError

In [ ]:
# zgodnosc sasiadow: Jaccard 10 najblizszych sasiadow elementu wg TF-IDF i llm

sample = np.random.default_rng(PARAMS.seed).choice(len(items), min(500, len(items)), replace=False)
neighbour_jaccard = np.array([jaccard(most_similar(tfidf, i, 10), most_similar(llm, i, 10)) for i in sample])
print(f'srednio: {neighbour_jaccard.mean():.2f}')

# dla dwoch list 10-elementowych Jaccard przyjmuje tylko 11 wartosci, wiec slupki zamiast histogramu
values, counts = np.unique(neighbour_jaccard.round(3), return_counts=True)
plt.bar(values, counts, width=0.03)
plt.xlabel('Jaccard')
plt.ylabel('liczba elementów')
plt.title('Zgodność 10 najbliższych sąsiadów: TF-IDF vs llm')
plt.show()

In [ ]:
def compare(item: int, k: int = 5) -> pd.DataFrame:
    return pd.DataFrame({
        'tfidf': [label(i) for i in most_similar(tfidf, item, k)],
        'llm': [label(i) for i in most_similar(llm, item, k)],
    })


# przyklady z czesci 3. i dwa elementy, dla ktorych metody zgadzaja sie najmniej
# zauwaz: czy TF-IDF trzyma sie tych samych slow (marka, seria), a llm - znaczenia?
for item in EXAMPLE_ITEMS + list(sample[np.argsort(neighbour_jaccard)[:2]]):
    print(items.title[item])
    display(compare(item))

## Część 5. - rekomendacje dla użytkowników

In [ ]:
liked = train[train.rating >= PARAMS.like_threshold]
history: Dict[str, List[int]] = liked.groupby('user_id').item.apply(list).to_dict()  # lubiane - z nich budujemy profil
seen: Dict[str, List[int]] = train.groupby('user_id').item.apply(list).to_dict()  # wszystkie ocenione - wykluczamy je


# profil uzytkownika - srednia wektorow lubianych elementow, wektor 1 x liczba cech (dla tfidf i llm)

def user_profile(vectors, user_history: List[int]) -> np.ndarray:
    # vectors[lista_indeksow] wybiera wiersze, .mean(axis=0) dziala dla macierzy rzadkiej i gestej
    raise NotImplementedError


def recommend(vectors, user: str, k: int) -> List[int]:
    # profil z history[user], wykluczamy seen[user]
    raise NotImplementedError

In [ ]:
# baseline - najpopularniejsze elementy z ostatniego okresu historii

def popularity_scores(train: pd.DataFrame, days: int) -> np.ndarray:
    recent = train[train.timestamp >= train.timestamp.max() - days * 24 * 3600 * 1000]  # timestamp w ms
    return np.bincount(recent.item, minlength=len(items)).astype(float)


popularity = popularity_scores(train, PARAMS.popularity_days)


def recommend_popular(user: str, k: int) -> List[int]:
    return top_k(popularity, k, exclude=seen[user])


RECOMMENDERS: Dict[str, Callable[[str, int], List[int]]] = {
    'popularity': recommend_popular,
    'tfidf': lambda user, k: recommend(tfidf, user, k),
    'llm': lambda user, k: recommend(llm, user, k),
}

**Trafne** elementy użytkownika: polubione w teście i nieocenione wcześniej.
Oceniamy użytkowników z co najmniej `min_history` lubianymi elementami w historii.

In [ ]:
test_liked = test[test.rating >= PARAMS.like_threshold].groupby('user_id').item.apply(set).to_dict()
relevant: Dict[str, Set[int]] = {
    user: liked_items - set(seen[user])
    for user, liked_items in test_liked.items()
    if len(history.get(user, [])) >= PARAMS.min_history
}
relevant = {user: liked_items for user, liked_items in relevant.items() if liked_items}
eval_users = sorted(np.random.default_rng(PARAMS.seed).choice(sorted(relevant), min(PARAMS.eval_users, len(relevant)), replace=False))
print(len(relevant), len(eval_users))

In [ ]:
user = eval_users[0]  # zmien indeks, by zobaczyc innych uzytkownikow
print('historia:')
print_items(history[user])
print('test:')
print_items(relevant[user])
for name, recommender in RECOMMENDERS.items():
    print(f'{name}:')
    print_items(recommender(user, 10), relevant[user])


## Część 6. - ocena jakości

Dla listy top-$k$ i zbioru trafnych elementów $T$:

$$\text{Hit@}k = \begin{cases} 1 & \text{gdy } \text{top-}k \cap T \neq \emptyset \\ 0 & \text{w p.p.} \end{cases} \qquad \text{Recall@}k = \frac{|\text{top-}k \cap T|}{|T|}$$

 * `cold_recall@k` - Recall@k liczony tylko na nowych trafnych elementach
 * `coverage@k` - jaka część katalogu pojawia się w czyichkolwiek rekomendacjach
 * `cold@k` - udział nowych elementów w rekomendacjach

In [ ]:
def hit_rate(recommended: Sequence[int], relevant: Set[int]) -> float:
    raise NotImplementedError


def recall(recommended: Sequence[int], relevant: Set[int]) -> float:
    raise NotImplementedError

In [ ]:
def evaluate(recommenders, users: List[str], ks: Tuple[int, ...]) -> Tuple[pd.DataFrame, Dict[str, Dict[str, List[int]]]]:
    lists = {name: {user: rec(user, max(ks)) for user in users} for name, rec in recommenders.items()}
    cold_users = [user for user in users if relevant[user] & cold_items]
    rows = []
    for name, user_lists in lists.items():
        row = {'method': name}
        for k in ks:
            top = [user_lists[user][:k] for user in users]
            row[f'hit@{k}'] = np.mean([hit_rate(t, relevant[user]) for t, user in zip(top, users)])
            row[f'recall@{k}'] = np.mean([recall(t, relevant[user]) for t, user in zip(top, users)])
            row[f'cold_recall@{k}'] = np.mean([recall(user_lists[user][:k], relevant[user] & cold_items) for user in cold_users])
            row[f'coverage@{k}'] = len({i for t in top for i in t}) / len(items)
            row[f'cold@{k}'] = np.mean([i in cold_items for t in top for i in t])
        rows.append(row)
    return pd.DataFrame(rows).set_index('method'), lists


# niskie wartosci (pojedyncze %) sa tu typowe - trafic konkretny produkt sposrod ~25 tys. jest trudno
results, rec_lists = evaluate(RECOMMENDERS, eval_users, PARAMS.ks)
results.round(4).T

In [ ]:
k = min(PARAMS.ks)
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))

lengths = range(1, max(PARAMS.ks) + 1)
for name, user_lists in rec_lists.items():
    left.plot(lengths, [np.mean([recall(user_lists[u][:n], relevant[u]) for u in eval_users]) for n in lengths], label=name)
left.set_xlabel('k')
left.set_ylabel('Recall@k')
left.set_title('Trafność a długość listy')
left.legend()

for name in results.index:
    right.scatter(results.loc[name, f'coverage@{k}'], results.loc[name, f'recall@{k}'])
    right.annotate(name, (results.loc[name, f'coverage@{k}'], results.loc[name, f'recall@{k}']), xytext=(5, -12), textcoords='offset points')
right.set_xscale('log')
right.set_xlabel(f'coverage@{k} (skala log)')
right.set_ylabel(f'Recall@{k}')
right.set_title('Trafność a pokrycie katalogu')
plt.show()


In [ ]:
# zgodnosc list rekomendacji: Jaccard top-k list uzytkownika wg roznych metod (por. zgodnosc sasiadow z czesci 4.)

names = list(rec_lists)
overlap = pd.DataFrame(
    [[np.mean([jaccard(rec_lists[a][u][:k], rec_lists[b][u][:k]) for u in eval_users]) for b in names] for a in names],
    index=names, columns=names,
)
overlap.round(3)
